# Aleo: one search result and one detail record

This small playbook demonstrates the pipeline rather than crawling an entire catalogue. The search worker returns one company and schedules its detail page; the detail worker returns one enriched company.

In [3]:
import hashlib
import json
import os
from dataclasses import dataclass, field
from typing import Any
from urllib.parse import urljoin

import pandas as pd
from longscrape import (
    Crawler,
    DefaultExtractor,
    ExtractionResult,
    FetchRequest,
    PipelineInput,
    RawEntry,
    RichEntry,
    ScraperWorker,
)
from longscrape.adapters import DefaultFetcher, PatchrightManager, URLBlocklist
from longscrape.adapters.playwright.middlewares import URLCacher
from longscrape.adapters.store.raw_entry import PyMongoRawEntryStore
from parsel import Selector

In [4]:
SEARCH_KIND = "aleo.search"
DETAILS_KIND = "aleo.details"
ALEO_ORIGIN = "https://aleo.com/int/"
SEARCH_URL = (
    "https://aleo.com/int/companies/it-i-telekomunikacja/"
    "doradztwo-techniczne?voivodeships=LODZ&city=%C5%81%C3%B3d%C5%BA"
)


@dataclass
class Company:
    id: str
    name: str | None
    details_url: str
    source: str
    nip: str | None = None
    krs: str | None = None
    regon: str | None = None
    address: dict[str, Any] | None = None
    categories: list[list[str]] = field(default_factory=list)
    people_count: int | None = None


@dataclass
class Person:
    id: str
    name: str
    company_id: str
    age: int | None = None
    roles: list[dict[str, Any]] = field(default_factory=list)

In [5]:
def clean_text(values: list[str]) -> str | None:
    value = " ".join(part.strip() for part in values if part.strip())
    return value or None


def normalize_name(first_name: str | None, last_name: str | None) -> str:
    return " ".join(part for part in (first_name, last_name) if part).strip()


def find_api_response(state: dict[str, Any], suffix: str) -> dict[str, Any]:
    for response in state.values():
        if isinstance(response, dict) and suffix in response.get("u", ""):
            return response.get("b", {})
    return {}

In [6]:
class AleoSearchExtractor(DefaultExtractor[Company]):
    """Extract the first search card and schedule its single detail page."""

    def __init__(self) -> None:
        super().__init__(allowed_domain="aleo.com")

    async def extract(
        self, input: PipelineInput, raw_entry: RawEntry
    ) -> ExtractionResult[Company]:
        card = Selector(text=raw_entry.text).css("app-base-catalog-row").get()
        if card is None:
            return ExtractionResult(items=[], tasks=[])

        selector = Selector(text=card)
        link = selector.css("a.catalog-row-first-line__company-name")
        href = link.attrib.get("href")
        if not href:
            return ExtractionResult(items=[], tasks=[])

        nip = clean_text(selector.css(".tax-id::text").getall())
        krs = clean_text(selector.css(".krs::text").getall())
        regon = clean_text(selector.css(".regon::text").getall())
        details_url = urljoin(ALEO_ORIGIN, href)
        company = Company(
            id=krs or regon or nip or href.rsplit("/", 1)[-1],
            name=clean_text(link.css("::text").getall()),
            details_url=details_url,
            source="search",
            nip=nip,
            krs=krs,
            regon=regon,
        )
        return ExtractionResult(
            items=[RichEntry(url=raw_entry.url, data=company)],
            tasks=[input.spawn(kind=DETAILS_KIND, query=details_url)],
        )


class AleoDetailsExtractor(DefaultExtractor[Company | Person]):
    """Extract one company and its associated people from its detail page."""

    def __init__(self) -> None:
        super().__init__(allowed_domain="aleo.com")

    async def extract(
        self, input: PipelineInput, raw_entry: RawEntry
    ) -> ExtractionResult[Company | Person]:
        page = Selector(text=raw_entry.text)
        state_json = page.css("script#ng-state::text").get()
        if not state_json:
            raise ValueError("Aleo detail page has no ng-state payload")

        state = json.loads(state_json)
        company_data = find_api_response(state, "/api/v2/companies")
        identification = company_data.get("identification", {})
        registry_data = company_data.get("registryData", {})
        company_id = (
            identification.get("krs")
            or identification.get("regon")
            or identification.get("nip")
            or identification.get("id")
            or hashlib.sha256(raw_entry.url.encode()).hexdigest()
        )

        relations = find_api_response(state, "/relations")
        people_by_name: dict[str, Person] = {}
        for node in relations.get("personalNodes", []):
            name = node.get("name")
            if not name:
                continue
            roles = [
                {"type": label.get("connectionType"), "label": label.get("label")}
                for relation in node.get("data", [])
                for label in relation.get("labels", [])
                if relation.get("companyId") == identification.get("id")
            ]
            people_by_name[name] = Person(
                id=str(node["id"]),
                name=name,
                company_id=str(company_id),
                age=node.get("age"),
                roles=roles,
            )

        registry_details = find_api_response(state, "krs-registry-data")
        for group, authorities in registry_details.get("authorities", {}).items():
            if not isinstance(authorities, list):
                continue
            for authority in authorities:
                name = normalize_name(
                    authority.get("firstName"), authority.get("lastName")
                )
                if not name:
                    continue
                person = people_by_name.setdefault(
                    name,
                    Person(
                        id=hashlib.sha256(f"{company_id}:{name}".encode()).hexdigest(),
                        name=name,
                        company_id=str(company_id),
                        age=authority.get("age"),
                    ),
                )
                person.roles.append(
                    {
                        "type": group,
                        "function": authority.get("function"),
                        "shares": authority.get("shares"),
                    }
                )

        categories = [
            [item["name"] for item in category.get("paths", []) if item.get("name")]
            for category in company_data.get("categories", [])
        ]
        company = Company(
            id=str(company_id),
            name=registry_data.get("name")
            or clean_text(page.css(".text-company-name::text").getall()),
            details_url=raw_entry.url,
            source="details",
            nip=identification.get("nip"),
            krs=identification.get("krs"),
            regon=identification.get("regon"),
            address=(company_data.get("addresses") or [{}])[0],
            categories=categories,
            people_count=len(people_by_name),
        )
        return ExtractionResult(
            items=[
                RichEntry(url=raw_entry.url, data=company),
                *(
                    RichEntry(url=raw_entry.url, data=person)
                    for person in people_by_name.values()
                ),
            ],
            tasks=[],
        )

## Browser and fetcher setup

The crawler manages both resources, so this cell only defines the browser-backed fetchers and workers.

In [7]:
playwright = PatchrightManager(headless=False)
playwright.register_middleware(URLBlocklist())
playwright.register_middleware(URLCacher())

fetcher = DefaultFetcher(playwright, "aleo.com")
raw_entries = PyMongoRawEntryStore(
    os.environ.get("MONGODB_URI", "mongodb://localhost:27017")
)
workers = {
    SEARCH_KIND: ScraperWorker(
        fetcher,
        AleoSearchExtractor(),
        task_kind=SEARCH_KIND,
        raw_entry_store=raw_entries,
    ),
    DETAILS_KIND: ScraperWorker(
        fetcher,
        AleoDetailsExtractor(),
        task_kind=DETAILS_KIND,
        raw_entry_store=raw_entries,
    ),
}

## Run and print

A single search request produces exactly one search entry and one detail entry.

In [8]:
async with Crawler(workers, resources=[playwright, raw_entries]) as crawler:
    entries = await crawler.run(FetchRequest(kind=SEARCH_KIND, query=SEARCH_URL))

for entry in entries:
    item = entry.data
    if isinstance(item, Company):
        print(f"[{item.source}] {item.name} — {item.details_url}")
    else:
        roles = ", ".join(
            sorted({role.get("type", "") for role in item.roles if role.get("type")})
        )
        print(f"[person] {item.name} ({roles or 'associated person'})")

[search] HUBNERBIT SPÓŁKA Z OGRANICZONĄ ODPOWIEDZIALNOŚCIĄ — https://aleo.com/int/company/hubnerbit-spolka-z-ograniczona-odpowiedzialnoscia
[details] HUBNERBIT SPÓŁKA Z OGRANICZONĄ ODPOWIEDZIALNOŚCIĄ — https://aleo.com/int/company/hubnerbit-spolka-z-ograniczona-odpowiedzialnoscia
[person] KAJETAN KAMIL HÜBNER (BOARD_MEMBER, SHAREHOLDER, boardMembers, shareholders)
[person] KATARZYNA ZOFIA PIERZGALSKA (BOARD_MEMBER, SHAREHOLDER, boardMembers, shareholders)


In [9]:
company_table = pd.DataFrame(
    {
        "source": item.source,
        "company_id": item.id,
        "name": item.name,
        "nip": item.nip,
        "krs": item.krs,
        "regon": item.regon,
        "detail_url": item.details_url,
        "people_count": item.people_count,
    }
    for entry in entries
    if isinstance(item := entry.data, Company)
)
people_table = pd.DataFrame(
    {
        "person_id": item.id,
        "name": item.name,
        "company_id": item.company_id,
        "age": item.age,
        "roles": "; ".join(
            sorted({role.get("type", "") for role in item.roles if role.get("type")})
        ),
    }
    for entry in entries
    if isinstance(item := entry.data, Person)
)
company_table, people_table

(    source  company_id                                               name  \
 0   search  0000669769  HUBNERBIT SPÓŁKA Z OGRANICZONĄ ODPOWIEDZIALNOŚCIĄ   
 1  details  0000669769  HUBNERBIT SPÓŁKA Z OGRANICZONĄ ODPOWIEDZIALNOŚCIĄ   
 
           nip         krs      regon  \
 0  7272812046  0000669769  366851409   
 1  7272812046  0000669769  366851409   
 
                                           detail_url  people_count  
 0  https://aleo.com/int/company/hubnerbit-spolka-...           NaN  
 1  https://aleo.com/int/company/hubnerbit-spolka-...           2.0  ,
      person_id                         name  company_id  age  \
 0   -327928928         KAJETAN KAMIL HÜBNER  0000669769   34   
 1  -2107509970  KATARZYNA ZOFIA PIERZGALSKA  0000669769   34   
 
                                                roles  
 0  BOARD_MEMBER; SHAREHOLDER; boardMembers; share...  
 1  BOARD_MEMBER; SHAREHOLDER; boardMembers; share...  )